## **Setup**

In [1]:
import os

os.environ["TF_USE_LEGACY_KERAS"] = "1"

import gdown
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import accuracy_score, recall_score, f1_score
from sklearn.metrics.pairwise import cosine_similarity
import xgboost as xgb

import tensorflow as tf
import tensorflow_recommenders as tfrs
from tensorflow.keras.layers import StringLookup, TextVectorization, Embedding, GRU, Dense
from tensorflow.keras import layers

2025-11-16 17:22:45.477639: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-16 17:22:45.765589: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-16 17:22:47.126746: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


## **Data preparation**

In [2]:
# Download data
file_id = "1GffOYmcAMP17oi2BwC7Dp4l5F7rEjHRr" 
url = f"https://drive.google.com/uc?id={file_id}"
output = "mind_large.zip"
is_downloaded = False

for file in os.listdir("."):
    if file.startswith("mind_large"):
        print("mind_large is already downloaded.")
        is_downloaded = True
        break
        
if is_downloaded == False:
    print("Downloading mind_large.zip ...")
    gdown.download(url, output, quiet=False)
    print("Download complete!")

    # Extract compressed file
    with zipfile.ZipFile("mind_large.zip", "r") as z:
        z.extractall(".")

mind_large is already downloaded.


In [3]:
# Sets
sets = ["train", "dev", "test"]

# News
news_header = ["id", "category", "subcategory", "title", "abstract", "url", "title_entities", "abstract_entities"]
news = {}
for _set in sets:
    news[_set] = pd.read_csv(f"mind_large/news_{_set}.tsv", names=news_header, sep="\t")
all_news_df = pd.concat([news["train"], news["dev"], news["test"]], ignore_index=True)    

# Impressions
behaviors_header = ["impression_id", "user_id", "time", "history", "impressions"]
behaviors = {}
for _set in sets:
    behaviors[_set] = pd.read_csv(f"mind_large/behaviors_{_set}.tsv", names=behaviors_header, sep="\t")
all_behaviors_df = pd.concat([behaviors["train"], behaviors["dev"], behaviors["test"]], ignore_index=True)    

In [4]:
all_news_df = all_news_df.drop(columns=["url", "title_entities", "abstract_entities"])
all_news_df.drop_duplicates(inplace=True)
all_news_df["abstract"] = all_news_df["abstract"].fillna(all_news_df["title"])

In [5]:
all_behaviors_df["history"] = all_behaviors_df["history"].fillna("")

## **Filtering data (to keep relevant rows)**

In [6]:
len(all_behaviors_df)

4979946

In [7]:
behaviors_df = all_behaviors_df[
    (all_behaviors_df["impressions"].str.contains("-1")) & # Keep only impressions with at least one click
    (all_behaviors_df["history"].str.len() > 0) & # Keep users with AT LEAST ONE item in their history
    (all_behaviors_df["impressions"].str.len() >= 10) # Keep only impressions with at least 2 items shown
]

len(behaviors_df)

2551884

In [8]:
# Sampling (my computer is not that performant :))
behaviors_df = behaviors_df.sample(n=20_000)

## **Building the user-news interaction dataset**

In [9]:
# Function to split the impressions and clicks into two separate lists
def process_impression(impression_list):
    clicked, non_clicked = [], []
    if impression_list != "":
        list_of_strings = impression_list.split()
        clicked = [x.split("-")[0] for x in list_of_strings if x.split("-")[1] == "1"]
        non_clicked = [x.split("-")[0] for x in list_of_strings if x.split("-")[1] == "0"]
    return clicked, non_clicked

In [10]:
# Separate views from clicks
behaviors_df[["clicked", "non_clicked"]] = behaviors_df["impressions"].apply(
    lambda x: pd.Series(process_impression(x))
)
# Split history
behaviors_df["history"] = behaviors_df["history"].apply(lambda x: x.split(" "))

behaviors_df.head()

,impression_id,user_id,time,history,impressions,clicked,non_clicked
1392114,1392115,U660280,11/11/2019 5:49:32 AM,"[N52652, N9740, N92085, N64593, N32645, N11805...",N71120-0 N2210-0 N57903-0 N33625-0 N116501-0 N...,"[N7551, N88083]","[N71120, N2210, N57903, N33625, N116501, N2248..."
2093356,2093357,U532425,11/11/2019 1:44:30 PM,"[N71340, N38742, N62401, N119778, N104498, N73...",N49168-0 N128881-0 N117260-0 N46945-0 N123234-...,[N4612],"[N49168, N128881, N117260, N46945, N123234, N9..."
757762,757763,U400878,11/11/2019 6:27:40 PM,"[N77001, N110755, N20311, N128643, N122927, N2...",N53398-1 N55792-0 N75391-0 N14925-0 N114227-1 ...,"[N53398, N114227, N85986]","[N55792, N75391, N14925, N71394, N115676, N822..."
1336456,1336457,U385342,11/12/2019 2:14:19 PM,"[N7154, N122359, N58295, N75175, N19347, N3948...",N72485-0 N74117-0 N49283-1,[N49283],"[N72485, N74117]"
2096555,2096556,U524483,11/11/2019 2:35:03 PM,"[N39049, N87437, N89611, N50489, N72571, N1041...",N77011-0 N123077-0 N31879-0 N38304-0 N67265-0 ...,"[N98178, N47257]","[N77011, N123077, N31879, N38304, N67265, N330..."


In [11]:
%%time

click_data = []
for _, row in behaviors_df.iterrows():
    history = row["history"]
    clicked_news, non_clicked_news = row["clicked"], row["non_clicked"]
    
    for news_id in clicked_news:
        click_data.append({
            "history": history,
            "candidate_news_id": news_id,
            "label": 1
        })
        
    # We can also sample non-clicked news for harder negatives
    for news_id in non_clicked_news:
         click_data.append({
            "history": history,
            "candidate_news_id": news_id,
            "label": 0
        })

# Create a DataFrame from the exploded data
training_df = pd.DataFrame(click_data)
training_df.head()

CPU times: user 1.89 s, sys: 76 ms, total: 1.96 s
Wall time: 1.96 s


,history,candidate_news_id,label
0,"[N52652, N9740, N92085, N64593, N32645, N11805...",N7551,1
1,"[N52652, N9740, N92085, N64593, N32645, N11805...",N88083,1
2,"[N52652, N9740, N92085, N64593, N32645, N11805...",N71120,0
3,"[N52652, N9740, N92085, N64593, N32645, N11805...",N2210,0
4,"[N52652, N9740, N92085, N64593, N32645, N11805...",N57903,0


In [ ]:
# For Retrieval (Two-Tower): We only need positive interactions (label=1)
retrieval_df = training_df[training_df["label"] == 1].copy()

# This will be used to join features to the candidate_news_id
news_features_df = all_news_df[["id", "category", "title"]].copy()
news_features_df = news_features_df.rename(columns={"id": "candidate_news_id"})

# Merge retrieval_df with all_news_df
retrieval_df_merged = retrieval_df.merge(
    news_features_df,
    on="candidate_news_id",
    how="left"
)

# We now use the merged DataFrame which contains all the required features.
# We also use the keys that compute_loss expects ("news_id", "category", "title")
retrieval_ds = tf.data.Dataset.from_tensor_slices({
    "history": tf.ragged.constant(retrieval_df_merged["history"].values),
    "news_id": tf.constant(retrieval_df_merged["candidate_news_id"].values),
    "category": tf.constant(retrieval_df_merged["category"].values),
    "title": tf.constant(retrieval_df_merged["title"].values)
})

print(f"Created {len(retrieval_df_merged)} positive pairs for retrieval training.")

Created 30115 positive pairs for retrieval training.


E0000 00:00:1763310216.393008   12754 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1763310216.405696   12754 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


## **Stage 1: Retrieval (Two-Tower Model)**

**Create vocabularies for categorical features**

In [13]:
all_news_ids = all_news_df["id"].unique()
all_categories = all_news_df["category"].unique()

**Defining the hyperparameters for the training**

In [14]:
EMBEDDING_DIM = 64 # Output embeddings dimension for both User and News tower
MAX_HISTORY_LENGTH = 30 # Max number of articles to look at in user history
MAX_TOKENS = 20000 # Max vocab size for titles
TITLE_VECTORIZATION_DIM = 100

**Preprocessing and Vectorization**

In [15]:
# Define Preprocessing Layers (Vocabularies)
news_id_lookup = StringLookup(vocabulary=all_news_ids, mask_token=None)
category_lookup = StringLookup(vocabulary=all_categories, mask_token=None)

# Title tokenizer
title_vectorizer = TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode="int",
    output_sequence_length=TITLE_VECTORIZATION_DIM
)
# Adapt the layer to the news titles
title_vectorizer.adapt(all_news_df["title"])

**Building the News Tower**

This model turns a NewsID into an embedding

In [16]:
class NewsModel(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.news_id_embedding = Embedding(input_dim=len(all_news_ids) + 1, output_dim=EMBEDDING_DIM)
        self.category_embedding = Embedding(input_dim=len(all_categories) + 1, output_dim=EMBEDDING_DIM)
        self.title_embedding = tf.keras.Sequential([
            tf.keras.Input(shape=(1,), dtype=tf.string),
            title_vectorizer,
            Embedding(input_dim=MAX_TOKENS, output_dim=EMBEDDING_DIM),
            tf.keras.layers.GlobalAveragePooling1D() # Average word embeddings
        ])
        
        # Combine embeddings into a final dense layer
        self.dense = Dense(EMBEDDING_DIM)
        
        # Store news data for quick lookup
        self.news_data = tf.data.Dataset.from_tensor_slices({
            "news_id": all_news_df["id"].values,
            "category": all_news_df["category"].values,
            "title": all_news_df["title"].values
        }).batch(128)
    
    
    def call(self, inputs):
        # "inputs" is a dictionary: {"news_id": ..., "category": ..., "title": ...}
        
        # 1. Get embedding for each feature
        id_embedding = self.news_id_embedding(news_id_lookup(inputs["news_id"]))
        cat_embedding = self.category_embedding(category_lookup(inputs["category"]))
        title_embedding_vec = self.title_embedding(inputs["title"])
        
        # 2. Combine them
        combined_embeddings = tf.concat([id_embedding, cat_embedding, title_embedding_vec], axis=1)
        
        # 3. Pass through the final dense layer to get a single 64-dim vector
        return self.dense(combined_embeddings)

**Build the User (Session) Tower**

This model turns a user's click history into an embedding

In [17]:
class UserModel(tf.keras.Model):
    def __init__(self, news_id_embedding_layer):
        super().__init__()
        # Use the *same* embedding layer as the news model
        # This is a form of transfer learning
        
        self.news_id_embedding = news_id_embedding_layer
        # self.news_id_embedding = Embedding(input_dim=len(all_news_ids) + 1, output_dim=EMBEDDING_DIM)
        
        # We use a GRU to process the sequence of clicked news
        self.gru = GRU(EMBEDDING_DIM)
        # We could also just average:
        # self.pool = tf.keras.layers.GlobalAveragePooling1D()

    def call(self, history):
        # "history" is a RaggedTensor of NewsIDs [batch_size, num_clicks]
        history = history[:, -MAX_HISTORY_LENGTH:]        
        history_int = news_id_lookup(history)
        
        # Get embeddings for each news ID in the history
        history_embeddings = self.news_id_embedding(history_int)
        
        return self.gru(history_embeddings)

**Combining the two towers**

In [18]:
class MINDRetrievalModel(tfrs.Model):
    def __init__(self, user_model, news_model):
        super().__init__()
        self.user_model = user_model
        self.news_model = news_model

        # This task computes the loss and metrics (e.g., factorized_top_k)
        # It uses in-batch negatives by default
        self.task = tfrs.tasks.Retrieval(
            metrics=tfrs.metrics.FactorizedTopK(
                # candidates=news_ds.batch(128).map(self.news_model)
                candidates=news_model.news_data.map(self.news_model)
            )
        )

    
    def compute_loss(self, features, training=False):
                
        user_embeddings = self.user_model(features["history"])
        
        # Create the dictionary for the news model
        news_features = {
            "news_id": features["news_id"], # Assuming you renamed 'candidate_news_id'
            "category": features["category"],
            "title": features["title"]
        }
        positive_news_embeddings = self.news_model(news_features)
        
        return self.task(user_embeddings, positive_news_embeddings)

**Training the model**

In [21]:
# news_model = NewsModel()
# user_model = UserModel(news_model.news_id_embedding)

# retrieval_model = MINDRetrievalModel(user_model, news_model)

# # Define the optimizer
# retrieval_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001))

# # Batch and cache the training data
# retrieval_ds_batched = retrieval_ds.shuffle(20_000).batch(32).cache()

# # Train for a few epochs
# retrieval_model.fit(retrieval_ds_batched,
#                     epochs=10)

# # Save the models for inference (we need the user tower to get user embeddings and the news tower to build the candidate index)
# # user_model.save("models/user_retrieval_model")
# # news_model.save("models/news_retrieval_model")